In [1]:
# !pip install --upgrade yfinance

In [2]:
#!pip install ipywidgets

In [6]:
# !pip install --upgrade notebook ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.5 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.5 MB 1.4 MB/s eta 0:00:04
   ----- ---------------------------------- 0.8/5.5 MB 1.4 MB/s eta 0:00:04
   ------- -------------------------------- 1.0/5.5 MB 1.5 MB/s eta 0:00:03
   ----------- ---------------------------- 1.6/5.5 MB 1.6 MB/s eta 0:00:03
   ------------- -------------------------- 1.8/5.5 MB 1.6 MB/s eta 0:00:03
   --------------- ------------------------ 2.1/5.5 MB 1.5 MB/s eta 0:00:03
   --------------- ------------------------ 2.1/5.5 MB 1.5 MB/s eta 0:00:03
   ----------------- ---------------------- 2.4/5.5 MB 1.3 MB/s eta 0:00:03
   ------------------ --------------------- 2.6/5.5 MB 1.3 MB/s eta 0:00:03
   -------------------- ------------------- 2.9/5.5 MB 1.3 MB/s eta 0:00:02
   -------------------- -----------

  You can safely remove it manually.


In [1]:
# Configuración para gráficos interactivos dentro del notebook
import plotly.io as pio
pio.renderers.default = 'notebook'

import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Output
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")
print("ℹ️  Si los widgets no se ven, ejecuta en TERMINAL: jupyter nbextension enable --py widgetsnbextension")

✅ Librerías cargadas correctamente
ℹ️  Si los widgets no se ven, ejecuta en TERMINAL: jupyter nbextension enable --py widgetsnbextension


In [2]:
def crear_dashboard_accion(ticker='AAPL', periodo='6mo', intervalo='1d'):
    """
    Descarga datos de yfinance y genera gráfico de velas + volumen.
    Maneja cualquier error y devuelve None si no hay datos.
    """
    try:
        # Descargar datos
        df = yf.download(ticker, period=periodo, interval=intervalo, progress=False)
        
        # Verificar que el DataFrame no esté vacío y tenga las columnas necesarias
        if df is None or len(df) == 0:
            print(f"⚠️ No hay datos para '{ticker}'. Verifica el símbolo.")
            return None
        
        # Asegurar que las columnas existan
        required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️ El DataFrame no tiene las columnas esperadas para '{ticker}'.")
            return None
        
        # Extraer arrays numpy para acceder a valores de forma segura
        open_vals = df['Open'].values
        high_vals = df['High'].values
        low_vals = df['Low'].values
        close_vals = df['Close'].values
        volume_vals = df['Volume'].values
        dates = df.index.values  # fechas como objetos datetime
        
        # Calcular medias móviles (usando pandas, pero si falla, las omitimos)
        try:
            ma50 = df['Close'].rolling(window=50).mean().values
            ma200 = df['Close'].rolling(window=200).mean().values
        except:
            ma50 = np.full(len(close_vals), np.nan)
            ma200 = np.full(len(close_vals), np.nan)
        
        # Obtener último precio y variación (usando valores nativos)
        ultimo = float(close_vals[-1])
        if len(close_vals) > 1:
            anterior = float(close_vals[-2])
        else:
            anterior = ultimo
        
        if anterior != 0:
            cambio = ((ultimo - anterior) / anterior) * 100
        else:
            cambio = 0.0
        
        max_price = float(np.max(high_vals)) if len(high_vals) > 0 else 0.0
        
        # Crear figura con dos filas
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            vertical_spacing=0.08, row_heights=[0.7, 0.3])
        
        # Velas Japonesas (usando arrays nativos)
        fig.add_trace(go.Candlestick(
            x=dates,
            open=open_vals,
            high=high_vals,
            low=low_vals,
            close=close_vals,
            name='OHLC'
        ), row=1, col=1)
        
        # Medias móviles (solo si hay datos suficientes)
        if len(ma50) > 0 and not np.isnan(ma50).all():
            fig.add_trace(go.Scatter(x=dates, y=ma50,
                                     line=dict(color='orange', width=1.5),
                                     name='MA50'), row=1, col=1)
        if len(ma200) > 0 and not np.isnan(ma200).all():
            fig.add_trace(go.Scatter(x=dates, y=ma200,
                                     line=dict(color='green', width=1.5),
                                     name='MA200'), row=1, col=1)
        
        # Volumen con colores (usamos arrays nativos para comparar)
        # Determinamos si cada vela es alcista o bajista comparando Close vs Open de cada fila
        colores = []
        for i in range(len(open_vals)):
            if close_vals[i] < open_vals[i]:
                colores.append('red')
            else:
                colores.append('green')
        
        fig.add_trace(go.Bar(x=dates, y=volume_vals,
                             name='Volumen',
                             marker_color=colores), row=2, col=1)
        
        # Mejoras estéticas
        fig.update_layout(
            template='plotly_dark',
            height=700,
            title=f'📈 {ticker} - Precio y Volumen',
            xaxis_rangeslider_visible=False
        )
        fig.update_yaxes(title_text='Precio (USD)', row=1, col=1)
        fig.update_yaxes(title_text='Volumen', row=2, col=1)
        
        # Mostrar métricas en consola
        print(f"💰 {ticker} -> Precio: ${ultimo:.2f}  |  Variación: {cambio:.2f}%  |  Máximo: ${max_price:.2f}")
        
        return fig
    
    except Exception as e:
        print(f"❌ Error inesperado en '{ticker}': {str(e)}")
        return None

In [3]:
# Controles
ticker_w = widgets.Text(value='AAPL', description='Ticker:', style={'description_width': 'initial'})
periodo_w = widgets.Dropdown(options=['1mo', '3mo', '6mo', '1y', '2y', '5y'], value='6mo', description='Período:')
intervalo_w = widgets.Dropdown(options=['1d', '1wk', '1mo'], value='1d', description='Intervalo:')

# Output
out = Output()

def actualizar_dashboard(ticker, periodo, intervalo):
    with out:
        out.clear_output(wait=True)
        fig = crear_dashboard_accion(ticker.strip().upper(), periodo, intervalo)
        if fig is not None:
            fig.show()
        else:
            print("🔍 Revisa que el ticker sea válido (ej: AAPL, MSFT, TSLA, GOOG)")
            print("🔄 Si el ticker es correcto, prueba con otro período o intervalo.")

# Conectar
widgets.interactive(actualizar_dashboard,
                    ticker=ticker_w,
                    periodo=periodo_w,
                    intervalo=intervalo_w)

# Mostrar
display(VBox([HBox([ticker_w, periodo_w, intervalo_w]), out]))